# Sentiment Analysis of IMDb Movie Reviews Using Apache Spark and Neural Networks

the main goal of this notebook is use Spark and Neural Networks to analyse movie reviews using a dataset from IMDb.

## Importing and installing necessary items/packages

In [1]:
%pip install -q pyspark scikit-learn pandas numpy matplotlib scipy

In [2]:

import os
import re
import time
import tarfile
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix

from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Starting Apache

In [3]:

spark = (
    SparkSession.builder
    .appName("IMDb-Sentiment-Spark-MLP")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)


Spark version: 4.0.4
Default parallelism: 2


## Upload the dataset
The dataset contains 50,000 labelled movie reviews:
- 25,000 training reviews
- 25,000 test reviews
- balanced positive and negative classes
- neutral reviews excluded

I'm having issues with Anaconda, so I'm using Google Collab for this demo (this is why i'm not using the dataset locally.

In [4]:

DATA_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
ARCHIVE = "aclImdb_v1.tar.gz"
DATA_DIR = "aclImdb"

if not os.path.exists(DATA_DIR):
    if not os.path.exists(ARCHIVE):
        print("Downloading IMDb dataset...")
        urllib.request.urlretrieve(DATA_URL, ARCHIVE)
    print("Extracting dataset...")
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        tar.extractall(".")
else:
    print("Dataset already available.")

print("Dataset path:", os.path.abspath(DATA_DIR))


Extracting dataset...
Dataset path: /content/aclImdb


## Load the train/test with Spark

In [5]:

def load_imdb_split(split_name):
    pos_path = f"{DATA_DIR}/{split_name}/pos/*.txt"
    neg_path = f"{DATA_DIR}/{split_name}/neg/*.txt"

    pos = spark.read.text(pos_path, wholetext=True).withColumn("label", lit(1))
    neg = spark.read.text(neg_path, wholetext=True).withColumn("label", lit(0))

    return pos.unionByName(neg).withColumnRenamed("value", "text")

train_df = load_imdb_split("train").cache()
test_df = load_imdb_split("test").cache()

print("Training rows:", train_df.count())
print("Test rows:", test_df.count())

print("\nTraining class distribution:")
train_df.groupBy("label").count().orderBy("label").show()

print("Test class distribution:")
test_df.groupBy("label").count().orderBy("label").show()


Training rows: 25000
Test rows: 25000

Training class distribution:
+-----+-----+
|label|count|
+-----+-----+
|    0|12500|
|    1|12500|
+-----+-----+

Test class distribution:
+-----+-----+
|label|count|
+-----+-----+
|    0|12500|
|    1|12500|
+-----+-----+



### The step above took a while to process. About 15 minutes